In [ ]:
!pip install pandas

In [ ]:
!pip install scikit-learn

In [ ]:
!pip install scikit-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
df=pd.read_csv(r'C:\Users\shali\Downloads\Dataset-SA.csv')

In [ ]:
print(df.head())

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
texts = df['Review'].values  
labels = df['Sentiment'].values  

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)


In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)

In [ ]:
print(train_texts[:10])  
print(type(train_texts))  
print(type(train_texts[0]))

In [ ]:
import numpy as np
train_texts = np.array([str(text) if pd.notna(text) else '' for text in train_texts])
print(train_texts[:10]) 

In [ ]:
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(train_texts) 

In [ ]:
train_texts = [str(text) for text in train_texts]
val_texts = [str(text) for text in val_texts]
train_sequences = tokenizer.texts_to_sequences(train_texts)
val_sequences = tokenizer.texts_to_sequences(val_texts)

In [ ]:
max_length = 100
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
val_padded = pad_sequences(val_sequences, maxlen=max_length, padding='post', truncating='post')


In [ ]:
import tensorflow as tf


In [ ]:
train_labels = tf.keras.utils.to_categorical(train_labels, num_classes=3)
val_labels = tf.keras.utils.to_categorical(val_labels, num_classes=3)

In [ ]:
from tensorflow.keras import layers, models

In [ ]:
model = models.Sequential([
    layers.Embedding(input_dim=10000, output_dim=64, input_length=max_length),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='softmax') 
])

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',  
              metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
history = model.fit(train_padded, train_labels, 
                    validation_data=(val_padded, val_labels),
                    epochs=10,  
                    batch_size=32) 

In [ ]:
val_loss, val_accuracy = model.evaluate(val_padded, val_labels)
print(f"Validation Accuracy: {val_accuracy*100:.2f}%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
epochs = np.arange(len(history.history['accuracy']))

In [ ]:
width = 0.35  
plt.bar(epochs - width/2, history.history['accuracy'], width=width, label='Training Accuracy')
plt.bar(epochs + width/2, history.history['val_accuracy'], width=width, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.bar(epochs - width/2, history.history['loss'], width=width, label='Training Loss')
plt.bar(epochs + width/2, history.history['val_loss'], width=width, label='Validation Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:

new_text = ["This product is fantastic!"]

new_sequences = tokenizer.texts_to_sequences(new_text)
new_padded = pad_sequences(new_sequences, maxlen=max_length, padding='post', truncating='post')


prediction = model.predict(new_padded)
predicted_label = label_encoder.inverse_transform([prediction.argmax(axis=-1)[0]])

print(f"Predicted Sentiment: {predicted_label[0]}")